In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [ ]:
bronze_path = "tihim_project.bronze.orders"
silver_path = "tihim_project.silver.orders"
quarantine_path = "tihim_project.silver.orders_quarantine"
checkpoint_path = "/Volumes/tihim_project/ops/stream_state/checkpoints/silver/orders"

MAX_QUANTITY = 1000
MAX_AMOUNT = 1000000

In [ ]:
spark.sql(f"""

    CREATE TABLE IF NOT EXISTS {silver_path} (

        order_id string,
        customer_id string,
        product_id string,
        order_date date,
        quantity int,
        total_amount double,
        unit_price double,
        ingestion_date timestamp,
        create_date timestamp,
        update_date timestamp
    )
    USING DELTA
    TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )      
        
    """)

In [ ]:
def clean_orders(df):
    
    return (
            df.drop("_rescued_data", "source_file")
            .withColumn("quantity", F.col("quantity").cast("int"))
            .withColumn("order_date", F.to_date(F.col("order_date")))
            .withColumn("total_amount", F.round(F.col("total_amount").cast("double"), 2))
            .withColumn("unit_price", F.when(F.col("quantity") > 0, F.round(F.col("total_amount") / F.col("quantity"), 2)).otherwise(F.lit(None).cast("double")))
            )

In [ ]:
def flag_orders(df):
    reasons = F.concat_ws("; ", F.when(F.col("order_id").isNull(), F.lit("order_id")),
                          F.when(F.col("customer_id").isNull(), F.lit("customer_id")),
                          F.when(F.col("product_id").isNull(), F.lit("product_id")),
                          F.when((F.col("quantity").isNull()) | (F.col("quantity") < 0), F.lit("invalid quantity")),
                          F.when(F.col("quantity") > MAX_QUANTITY, F.lit("quantity exceeds ceiling")),
                          F.when((F.col("total_amount").isNull()) | (F.col("total_amount") < 0), F.lit("invalid total_amount")),
                          F.when(F.col("total_amount") > MAX_AMOUNT, F.lit("total_amount exceeds ceiling")),
                          F.when(F.col("order_date").isNull(), F.lit("null order_date")),
                          F.when(F.col("order_date") > F.current_date(), F.lit("future order_date"))       
        )
    return df.withColumn("dq_reason", reasons)

In [ ]:
def upsert_orders_to_silver(microBatchDf, batchId):

    flagged = flag_orders(clean_orders(microBatchDf))

    valid = flagged.filter(F.col("dq_reason") == "")
    rejects = flagged.filter(F.col("dq_reason") != "")

    if not rejects.isEmpty():
        (rejects
            .withColumn("batch_id", F.lit(batchId))
            .withColumn("rejected_at", F.current_timestamp())
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(quarantine_path))

    w = Window.partitionBy("order_id").orderBy(F.col("ingestion_date").desc())
    final = (valid
        .withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1)
        .drop("rn", "dq_reason")
        .withColumn("create_date", F.col("ingestion_date"))
        .withColumn("update_date", F.col("ingestion_date")))
    

    try:
        (DeltaTable.forName(spark, silver_path).alias("target").merge(
            source = final.alias("update"),
            condition = "target.order_id = update.order_id"
        ).whenMatchedUpdate(
            condition = """
                NOT (target.customer_id  <=> update.customer_id)  OR
                NOT (target.product_id   <=> update.product_id)   OR
                NOT (target.quantity     <=> update.quantity)     OR
                NOT (target.unit_price   <=> update.unit_price)   OR
                NOT (target.total_amount <=> update.total_amount) OR
                NOT (target.order_date   <=> update.order_date)
            """,
            set = {
                "customer_id":  "update.customer_id",
                "product_id":   "update.product_id",
                "quantity":     "update.quantity",
                "unit_price":   "update.unit_price",
                "total_amount": "update.total_amount",
                "order_date":   "update.order_date",
                "update_date":  "update.update_date"
            }
        ).whenNotMatchedInsert(
            values = {
                "order_id":       "update.order_id",
                "customer_id":    "update.customer_id",
                "product_id":     "update.product_id",
                "order_date":     "update.order_date",
                "quantity":       "update.quantity",
                "unit_price":     "update.unit_price",
                "total_amount":   "update.total_amount",
                "ingestion_date": "update.ingestion_date",
                "create_date":    "update.create_date",
                "update_date":    "update.update_date"
            }
        ).execute())
    except Exception as e:
        print(f"[silver_orders] batch {batchId} failed: {e}")
        raise

    print(f"[silver_orders] batch {batchId}: {final.count()} valid, {rejects.count()} quarantined")


In [ ]:
query = (
    spark.readStream.table(bronze_path)
    .writeStream.foreachBatch(upsert_orders_to_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
    )

query.awaitTermination()